# Reading NetCDF File for Aruba (pH / Biogeochemistry)

This notebook demonstrates how to:
1. Load the Aruba pH NetCDF dataset (`CMVarubaph.nc`)
2. Convert it into a Pandas DataFrame containing `ph`, `dissic` (Dissolved Inorganic Carbon), and `talk` (Total Alkalinity)
3. Display summary statistics and visualize distributions for the entire dataset
4. Plot the monthly distributions of pH, Dissic, and Talk across all 48 months in the dataset
5. Subset the dataset into years (`year1`, `year2`, `year3`, and an 11-month `year4`) starting from the initial timestamp
6. Display summary statistics and visualize distributions for individual years

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### 1. Load the NetCDF file
We use `xarray.open_dataset` to load the `CMVarubaph.nc` file efficiently.

In [ ]:
nc_file = 'CMVarubaph.nc'
ds = xr.open_dataset(nc_file)
ds

### 2. Convert to DataFrame
We convert the dataset to a Pandas DataFrame and reset its index to flatten it.

In [ ]:
# Convert to DataFrame
df = ds.to_dataframe().reset_index()
print('DataFrame loaded successfully!')
df[['time', 'latitude', 'longitude', 'ph', 'dissic', 'talk']].head()

### 3. Inspect DataFrame Info & Summary Statistics (Entire Dataset)
View structural info and statistical summaries for pH, Dissic, and Talk across the entire Aruba dataset.

In [ ]:
print('--- DataFrame Info ---')
df.info()

print('\n--- Summary Statistics (Entire Dataset) ---')
display(df[['ph', 'dissic', 'talk']].describe())

### 4. Visualize Distributions (Entire Dataset)
Plot the histograms with clean decimal tick marks and units for pH, Dissic, and Talk.

In [ ]:
# Set visual style
sns.set_theme(style='whitegrid')

# Create side-by-side plots for all three variables
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. pH Histogram
sns.histplot(data=df, x='ph', bins=50, kde=True, ax=axes[0], color='#2c3e50')
axes[0].set_title('Entire Dataset - pH Distribution', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel('pH', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_xlim(7.95, 8.1)
axes[0].set_xticks([7.95, 8.0, 8.05, 8.1])

# 2. Dissic Histogram
sns.histplot(data=df, x='dissic', bins=50, kde=True, ax=axes[1], color='#16a085')
axes[1].set_title('Entire Dataset - Dissic Distribution', fontsize=14, fontweight='bold', pad=12)
axes[1].set_xlabel('Dissolved Inorganic Carbon (mol/m³)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_xlim(1.95, 2.15)
axes[1].set_xticks([1.95, 2.0, 2.05, 2.1, 2.15])

# 3. Talk Histogram
sns.histplot(data=df, x='talk', bins=50, kde=True, ax=axes[2], color='#e67e22')
axes[2].set_title('Entire Dataset - Talk Distribution', fontsize=14, fontweight='bold', pad=12)
axes[2].set_xlabel('Total Alkalinity (mol/m³)', fontsize=12)
axes[2].set_ylabel('Frequency', fontsize=12)
axes[2].set_xlim(2.3, 2.5)
axes[2].set_xticks([2.3, 2.35, 2.4, 2.45, 2.5])

plt.tight_layout()
plt.show()

### 5. Monthly Distribution of pH, Dissic, and Talk (Entire Dataset)
Box plots of speed and direction grouped by calendar month to show seasonal trends chronologically across all 48 months.

In [ ]:
# Create a sorted Year-Month string column for chronological grouping
df['year_month'] = df['time'].dt.strftime('%Y-%m')
sorted_months = sorted(df['year_month'].unique())

# Setup figure with 3 rows (pH, Dissic, and Talk)
fig, axes = plt.subplots(3, 1, figsize=(18, 18))

# 1. Box plot for pH
sns.boxplot(data=df, x='year_month', y='ph', ax=axes[0], palette='crest', order=sorted_months)
axes[0].set_title('Monthly Distribution of pH (Aruba)', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel('Month (Year-Month)', fontsize=12)
axes[0].set_ylabel('pH', fontsize=12)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].set_ylim(7.95, 8.1)
axes[0].set_yticks([7.95, 8.0, 8.05, 8.1])

# 2. Box plot for Dissic
sns.boxplot(data=df, x='year_month', y='dissic', ax=axes[1], palette='viridis', order=sorted_months)
axes[1].set_title('Monthly Distribution of Dissolved Inorganic Carbon (Aruba)', fontsize=14, fontweight='bold', pad=12)
axes[1].set_xlabel('Month (Year-Month)', fontsize=12)
axes[1].set_ylabel('Dissic (mol/m³)', fontsize=12)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_ylim(1.95, 2.15)
axes[1].set_yticks([1.95, 2.0, 2.05, 2.1, 2.15])

# 3. Box plot for Talk
sns.boxplot(data=df, x='year_month', y='talk', ax=axes[2], palette='flare', order=sorted_months)
axes[2].set_title('Monthly Distribution of Total Alkalinity (Aruba)', fontsize=14, fontweight='bold', pad=12)
axes[2].set_xlabel('Month (Year-Month)', fontsize=12)
axes[2].set_ylabel('Talk (mol/m³)', fontsize=12)
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha='right')
axes[2].set_ylim(2.3, 2.5)
axes[2].set_yticks([2.3, 2.35, 2.4, 2.45, 2.5])

plt.tight_layout()
plt.show()

### 6. Subset the Dataset into Years
Divide the dataset into chronological 12-month periods from the start timestamp:
- **`year1`**: July 5, 2022 – July 5, 2023 (12 months)
- **`year2`**: July 5, 2023 – July 5, 2024 (12 months)
- **`year3`**: July 5, 2024 – July 5, 2025 (12 months)
- **`year4`**: July 5, 2025 – June 4, 2026 (remaining 11 months)

In [ ]:
# Ensure time column is datetime objects
df['time'] = pd.to_datetime(df['time'])

# Define start date of the dataset
start_date = df['time'].min()

# Define 12-month boundary offsets
y1_end = start_date + pd.DateOffset(years=1)
y2_end = start_date + pd.DateOffset(years=2)
y3_end = start_date + pd.DateOffset(years=3)

# Subset into distinct copies
year1 = df[(df['time'] >= start_date) & (df['time'] < y1_end)].copy()
year2 = df[(df['time'] >= y1_end) & (df['time'] < y2_end)].copy()
year3 = df[(df['time'] >= y2_end) & (df['time'] < y3_end)].copy()
year4 = df[(df['time'] >= y3_end)].copy()  # Spans the remaining 11 months

# Print range and shapes
years_dfs = [year1, year2, year3, year4]
for i, y_df in enumerate(years_dfs, 1):
    label = f'Year {i}' if i < 4 else 'Year 4 (11 months)'
    print(f'{label}: {y_df["time"].min()} to {y_df["time"].max()} | Shape: {y_df.shape}')

### 7. Summary Statistics by Year
Descriptive statistics for each subsetted year.

In [ ]:
for i, y_df in enumerate(years_dfs, 1):
    label = f'Year {i}' if i < 4 else 'Year 4 (11-Month remaining)'
    print(f'\n{"="*20} {label} Summary Statistics {"="*20}')
    display(y_df[['ph', 'dissic', 'talk']].describe())

### 8. Visualizing Year-by-Year Distributions with Histograms
Compare pH, Dissic, and Talk distributions across all 4 years in a 4x3 subplot grid.

In [ ]:
# Set visual style
sns.set_theme(style='whitegrid')

# Create a 4x3 grid of subplots (4 years, 3 variables)
fig, axes = plt.subplots(4, 3, figsize=(18, 16))

colors_ph = ['#2c3e50', '#34495e', '#1a252f', '#2c3e50']
colors_dissic = ['#16a085', '#1abc9c', '#118f73', '#16a085']
colors_talk = ['#e67e22', '#f39c12', '#d35400', '#e67e22']

for i, y_df in enumerate(years_dfs):
    year_label = f'Year {i+1}' if i < 3 else 'Year 4 (11 Months)'
    
    # 1. pH Histograms (Left Column)
    sns.histplot(data=y_df, x='ph', bins=50, kde=True, ax=axes[i, 0], color=colors_ph[i])
    axes[i, 0].set_title(f'{year_label} - pH Distribution', fontsize=12, fontweight='bold')
    axes[i, 0].set_xlabel('ph', fontsize=10)
    axes[i, 0].set_ylabel('Frequency', fontsize=10)
    axes[i, 0].set_xlim(7.95, 8.1)
    axes[i, 0].set_xticks([7.95, 8.0, 8.05, 8.1])
    
    # 2. Dissic Histograms (Middle Column)
    sns.histplot(data=y_df, x='dissic', bins=50, kde=True, ax=axes[i, 1], color=colors_dissic[i])
    axes[i, 1].set_title(f'{year_label} - Dissic Distribution', fontsize=12, fontweight='bold')
    axes[i, 1].set_xlabel('Dissic (mol/m³)', fontsize=10)
    axes[i, 1].set_ylabel('Frequency', fontsize=10)
    axes[i, 1].set_xlim(1.95, 2.15)
    axes[i, 1].set_xticks([1.95, 2.0, 2.05, 2.1, 2.15])
    
    # 3. Talk Histograms (Right Column)
    sns.histplot(data=y_df, x='talk', bins=50, kde=True, ax=axes[i, 2], color=colors_talk[i])
    axes[i, 2].set_title(f'{year_label} - Talk Distribution', fontsize=12, fontweight='bold')
    axes[i, 2].set_xlabel('Talk (mol/m³)', fontsize=10)
    axes[i, 2].set_ylabel('Frequency', fontsize=10)
    axes[i, 2].set_xlim(2.3, 2.5)
    axes[i, 2].set_xticks([2.3, 2.35, 2.4, 2.45, 2.5])

plt.tight_layout()
plt.show()

### 9. Dataset Duration Verification
Calculates total temporal coverage.

In [ ]:
from dateutil.relativedelta import relativedelta

start_date = pd.to_datetime(df['time'].min())
end_date = pd.to_datetime(df['time'].max())
diff = relativedelta(end_date, start_date)

print(f'Start Date: {start_date}')
print(f'End Date:   {end_date}')
print(f'Total Duration: {diff.years} years, {diff.months} months, and {diff.days} days')